In [9]:
import pandas as pd

df = pd.read_excel(r"C:\Users\Hakan\Desktop\python ödev dashboard\B2B_Transaction_Data_.xlsx")
df.info()
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 664154 entries, 0 to 664153
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    664154 non-null  str    
 1   CustomerID   664154 non-null  str    
 2   InvoiceDate  664154 non-null  str    
 3   StockCode    664154 non-null  int64  
 4   Description  664154 non-null  object 
 5   Category     664090 non-null  str    
 6   City         657352 non-null  str    
 7   Quantity     664148 non-null  float64
 8   NetPrice     664154 non-null  float64
 9   UnitPrice    664154 non-null  float64
dtypes: float64(3), int64(1), object(1), str(5)
memory usage: 85.0+ MB


,InvoiceNo,CustomerID,InvoiceDate,StockCode,Description,Category,City,Quantity,NetPrice,UnitPrice
0,BSR202000000027,A02348,2020-01-02,8698543133198,Anatolian Puzzle Dünya Siyasi Harita (3319),1000 Parça Puzzle,ISTANBUL,1.0,18.92,22.8
1,BSR202000000007,A02212,2020-01-02,8698543110793,Anatolian-Puzzle 1000 Kurt Timi,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
2,BSR202000000019,A02212,2020-01-02,8698543110779,Anatolian-Puzzle 1000 Mustafa Kemal Atatürk,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
3,BSR202000000019,A02212,2020-01-02,8698543110779,Anatolian-Puzzle 1000 Mustafa Kemal Atatürk,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
4,BAD202000000028,A00905,2020-01-02,8698543110335,Gezegenler (Puzzle 1000) 1033,1000 Parça Puzzle,AYDIN,1.0,29.84,37.3
5,BSR202000000007,A02212,2020-01-02,8698543110366,Kartal Kayalıkları (Puzzle 1024) 1036,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
6,BSR202000000019,A02212,2020-01-02,8698543131170,Koydaki Bahçe (Puzzle 1000) 3117,1000 Parça Puzzle,ISTANBUL,2.0,29.84,37.3
7,BSR202000000019,A02212,2020-01-02,8698543131163,Mahşerin Dört Atı (Puzzle 1000) 3116,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
8,BSR202000000019,A02212,2020-01-02,8698543131163,Mahşerin Dört Atı (Puzzle 1000) 3116,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3
9,BSR202000000019,A02212,2020-01-02,8698543110151,Marilyn (Puzzle 1024) 1015,1000 Parça Puzzle,ISTANBUL,1.0,29.84,37.3


In [10]:
rapor = {"Ham satır sayısı": len(df)} ##kaç satırla uğraşıyoruz?

In [30]:

once = len(df)
df = df.drop_duplicates()
rapor["Silinen tekrar satır"] = once - len(df)
## tekrarlanan satırları sildik

In [31]:
rapor["Adet bilgisi eksik (silindi)"] = df["Quantity"].isna().sum()
df = df.dropna(subset=["Quantity"]) ##adet bilgisi olmadığı için hatalı bilgi vermeyi önledik.

In [32]:
rapor["Şehri bilinmeyen satır"] = df["City"].isna().sum()
rapor["Kategorisi bilinmeyen satır"] = df["Category"].isna().sum()
df["City"] = df["City"].fillna("Bilinmiyor")
df["Category"] = df["Category"].fillna("Bilinmiyor")
##herhangi bir satırın bir sütunda eksik verisi varsa ve bu hatalı gösterimlere sebep olucak bunu engellemek için bunu yaptık

In [33]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["NetPrice"]
df["Month"] = df["InvoiceDate"].dt.month
df["Weekday"] = df["InvoiceDate"].dt.dayofweek   # 0 = Pazartesi
## sütunlardaki verileri istediğimiz forma çevirir tarih revenue para olarak gibi

In [34]:
df["Description"] = df["Description"].astype(str)

In [35]:
rapor["Temiz satır sayısı"] = len(df)
rapor["Toplam ciro (TL)"] = round(df["Revenue"].sum(), 2)
df.to_parquet("b2b_clean.parquet", index=False)
pd.Series(rapor)
## gerekli temizlemeleri yaptıktan sonra excelde daha hızlı okunan parquet formatına çevirdik

Ham satır sayısı                  664154.00
Silinen tekrar satır                   0.00
Adet bilgisi eksik (silindi)           0.00
Şehri bilinmeyen satır                 0.00
Kategorisi bilinmeyen satır            0.00
Temiz satır sayısı                657935.00
Toplam ciro (TL)                29417403.65
dtype: float64

In [43]:

toplam_ciro = df["Revenue"].sum()
fatura_sayisi = df["InvoiceNo"].nunique()
musteri_sayisi = df["CustomerID"].nunique()
ort_fatura = toplam_ciro / fatura_sayisi
toplam_adet = df["Quantity"].sum()

print(f"Toplam ciro: {toplam_ciro:,.0f} TL")
print(f"Fatura sayısı: {fatura_sayisi:,}")
print(f"Aktif müşteri: {musteri_sayisi:,}")
print(f"Ortalama fatura tutarı: {ort_fatura:,.2f} TL")
print(f"Satılan adet: {toplam_adet:,.0f}")
##kpılar için gerekli hesaplamaları yapıyoruz.


Toplam ciro: 29,417,404 TL
Fatura sayısı: 47,471
Aktif müşteri: 1,202
Ortalama fatura tutarı: 619.69 TL
Satılan adet: 1,879,264


In [45]:
pd.Series(rapor).to_csv("veri_kalitesi.csv")